Load Era5 hourly data, pressure data for calculating wind shear

In [1]:
import sys
import xarray as xr
import matplotlib.pyplot as plt

sys.path.append("..")

from src.data.load_era5 import load_era5, convert_longitude, subset_area

# Load all three datasets
ds = load_era5("../data/raw/era5_sample_autumn2020")
ds_pressure = load_era5("../data/raw/era5_pressure_sample_autumn2020.nc")
ds_cape = load_era5("../data/raw/era5_cape_sample_autumn2020.nc")

# Clean up wind pressure 
ds_pressure = convert_longitude(ds_pressure)
ds_pressure = subset_area(ds_pressure, north=60, south=48, west=-10, east=2)

# Clean up cape
ds_cape = convert_longitude(ds_cape)
ds_cape = subset_area(ds_cape, north=60, south=48, west=-10, east=2)

# Merge everything into one dataset
ds_merged = xr.merge([ds, ds_pressure, ds_cape]).sortby("pressure_level", ascending=False)
print(ds_merged)

/home/alf_walks/projects/python/HPCWeatherPredictionPipeline/notebooks/../src/data/load_era5.py:9: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")
/home/alf_walks/projects/python/HPCWeatherPredictionPipeline/notebooks/../src/data/load_era5.py:9: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitl

<xarray.Dataset> Size: 178MB
Dimensions:         (valid_time: 488, latitude: 49, longitude: 49,
                     pressure_level: 8)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 4kB 2020-09-01 ... 2020-12-31...
    expver          (valid_time) <U4 8kB '0001' '0001' '0001' ... '0001' '0001'
  * latitude        (latitude) float64 392B 60.0 59.75 59.5 ... 48.5 48.25 48.0
  * longitude       (longitude) float64 392B -10.0 -9.75 -9.5 ... 1.5 1.75 2.0
  * pressure_level  (pressure_level) float64 64B 1e+03 925.0 ... 400.0 300.0
    number          int64 8B 0
Data variables:
    u10             (valid_time, latitude, longitude) float32 5MB dask.array<chunksize=(488, 49, 49), meta=np.ndarray>
    v10             (valid_time, latitude, longitude) float32 5MB dask.array<chunksize=(488, 49, 49), meta=np.ndarray>
    d2m             (valid_time, latitude, longitude) float32 5MB dask.array<chunksize=(488, 49, 49), meta=np.ndarray>
    t2m             (valid_time, latitude, longitu

/tmp/ipykernel_3070/719615568.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds, ds_pressure, ds_cape]).sortby("pressure_level", ascending=False)
/tmp/ipykernel_3070/719615568.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds, ds_pressure, ds_cape]).sortby("pressure_level", ascen

Calcuate bulk wind shear

In [ ]:
import sys
import xarray as xr
import matplotlib.pyplot as plt
sys.path.append("..")

from src.data.load_era5 import load_era5, convert_longitude, subset_area
from src.features.meteorology import compute_shear_features

# compute all shear features
shear_features = compute_shear_features(ds_merged)

Calculate Convective Available Potential Energy (CAPE)

In [ ]:
#OPTIONAL CELL
#Use for showing improvement on time taken with dask processes from sequential

import time
import numpy as np
import dask
from dask import delayed
from src.features.meteorology import (
    compute_cape_sequential,
    compute_cape_for_timestep,
)

n_time = len(ds_merged['valid_time'])

start = time.time()
result_seq = compute_cape_sequential(ds_merged)
time_seq = time.time() - start

start = time.time()
results_processes = dask.compute(
    *[delayed(compute_cape_for_timestep)(ds_merged, t) for t in range(n_time)],
    scheduler='processes'
)
time_processes = time.time() - start
cape_processes = np.stack(results_processes, axis=-1)

# Correctness 
match_processes = np.allclose(result_seq.values, cape_processes, equal_nan=True)

# Summary table
print(f"{'Method':<30}{'Time (s)':<12}{'Speedup':<10}{'Matches seq?'}")
print(f"{'Sequential':<30}{time_seq:<12.2f}{'1.00x':<10}{'-'}")
print(f"{'Dask batched (processes)':<30}{time_processes:<12.2f}{time_seq/time_processes:<10.2f}{match_processes}")

Method                        Time (s)    Speedup   Matches seq?
Sequential                    200.25      1.00x     -
Dask batched (processes)      30.98       6.46      True


In [3]:
from src.features.meteorology import compute_cape_dask_batched
xrCape = compute_cape_dask_batched(ds_merged.load())
print(xrCape)

/home/alf_walks/projects/python/HPCWeatherPredictionPipeline/.venv/lib/python3.12/site-packages/metpy/calc/thermo.py:1753: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)
/home/alf_walks/projects/python/HPCWeatherPredictionPipeline/.venv/lib/python3.12/site-packages/metpy/calc/thermo.py:1753: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)
/home/alf_walks/projects/python/HPCWeatherPredictionPipeline/.venv/lib/python3.12/site-packages/metpy/calc/thermo.py:1753: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)


<xarray.DataArray 'cape' (latitude: 49, longitude: 49, valid_time: 488)> Size: 9MB
array([[[0.00000000e+00, 5.32405276e+00, 1.04645983e+01, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 2.58067640e+00, 3.39498083e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 2.54329023e-02, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 2.25957002e+01, 1.57955291e+01],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 2.28728377e+01, 1.37937802e+01],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 2.41723877e+01, 9.79305120e+00]],

       [[0.00000000e+00, 6.39047150e+00, 1.39702329e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 3.87297222e+00, 0.00000000e+00, ...,


BRN


In [4]:
xrCape.to_netcdf("../data/processed/cape_autumn2020.nc")

In [6]:
import numpy as np

print("NaN count:", np.isnan(xrCape.values).sum())
print("Min:", xrCape.min().values, "Max:", xrCape.max().values)

NaN count: 0
Min: -204.10477812310617 Max: 1489.001388993977


In [7]:
from src.features.meteorology import compute_brn

brn = compute_brn(xrCape, shear_features["bulk_shear_10m_500"])
brn_named = brn.rename("brn")

precip = ds_merged['tp']

all_features = xr.merge([shear_features, xrCape, brn_named, precip])
dfFeatures = all_features.to_dataframe()
dfFeatures = dfFeatures.drop(columns=['number', 'expver'])
dfFeatures = dfFeatures.reset_index()

print(dfFeatures['tp'].describe())
print((dfFeatures['tp'] == 0).sum())

print("Number of NaN Values:")
print(dfFeatures.isna().sum())

print(dfFeatures['tp'].quantile([0.90, 0.95, 0.99]))
print((dfFeatures['tp'] >= 0.001).sum())  # how many rows exceed 1mm

dfFeatures['heavy_rain'] = (dfFeatures['tp'] >= 0.001).astype(int)
print("===============================")
print(dfFeatures['heavy_rain'].value_counts())

count    1.171688e+06
mean     1.659583e-04
std      3.857164e-04
min      0.000000e+00
25%      0.000000e+00
50%      1.668930e-05
75%      1.249313e-04
max      9.902954e-03
Name: tp, dtype: float64
312564
Number of NaN Values:
valid_time            0
latitude              0
longitude             0
bulk_shear_10m_850    0
bulk_shear_850_500    0
bulk_shear_10m_500    0
dir_shear_10m_850     0
dir_shear_850_500     0
dir_shear_10m_500     0
cape                  0
brn                   0
tp                    0
dtype: int64
0.90    0.000512
0.95    0.000906
0.99    0.001922
Name: tp, dtype: float64
49936
heavy_rain
0    1121752
1      49936
Name: count, dtype: int64


CHECKPOINT

In [8]:
#Checkpoint (so can start from here when restarting kernel)
dfFeatures.to_csv("../data/processed/era5_features_jan2020.csv", index=False)

In [9]:
from sklearn.model_selection import train_test_split

feature_columns = ["bulk_shear_10m_850", "bulk_shear_850_500", "bulk_shear_10m_500", "dir_shear_10m_850", "dir_shear_850_500", "dir_shear_10m_500", "cape", "brn"]
X = dfFeatures[feature_columns]
y = dfFeatures["heavy_rain"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y_train.value_counts())
print(y_test.value_counts())

heavy_rain
0    897401
1     39949
Name: count, dtype: int64
heavy_rain
0    224351
1      9987
Name: count, dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(pd.Series(y_pred).value_counts())

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0    163020
1     71318
Name: count, dtype: int64
Accuracy: 0.7075719686947913
[[159422  64929]
 [  3598   6389]]
              precision    recall  f1-score   support

           0       0.98      0.71      0.82    224351
           1       0.09      0.64      0.16      9987

    accuracy                           0.71    234338
   macro avg       0.53      0.68      0.49    234338
weighted avg       0.94      0.71      0.79    234338

